# Imports


In [ ]:
import keras
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
import tqdm.notebook as tqdm
import matplotlib.pyplot as plt

import os
import csv
import zipfile
import urllib.request


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except:
        pass


# Dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.


In [ ]:
# качаем официальные выгрузки Tatoeba: предложения eng/rus + связи rus-eng
import bz2

files = {
    'eng_sentences.tsv.bz2': 'https://downloads.tatoeba.org/exports/per_language/eng/eng_sentences.tsv.bz2',
    'rus_sentences.tsv.bz2': 'https://downloads.tatoeba.org/exports/per_language/rus/rus_sentences.tsv.bz2',
    'rus-eng_links.tsv.bz2': 'https://downloads.tatoeba.org/exports/per_language/rus/rus-eng_links.tsv.bz2',
}

os.makedirs('data', exist_ok=True)
for fname, url in files.items():
    out = 'data/' + fname[:-4]
    if os.path.exists(out):
        continue
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})  # без UA бывает 406
    with urllib.request.urlopen(req) as r, open('data/' + fname, 'wb') as f:
        f.write(r.read())
    with bz2.open('data/' + fname, 'rb') as fin, open(out, 'wb') as fout:
        fout.write(fin.read())
    print('скачали', out)


In [ ]:
eng = pd.read_csv('data/eng_sentences.tsv', sep='\t', header=None,
                  names=['id', 'lang', 'eng'], usecols=[0, 2], on_bad_lines='skip')
rus = pd.read_csv('data/rus_sentences.tsv', sep='\t', header=None,
                  names=['id', 'lang', 'rus'], usecols=[0, 2], on_bad_lines='skip')
links = pd.read_csv('data/rus-eng_links.tsv', sep='\t', header=None, names=['rus_id', 'eng_id'])

# джойним пары по id (eng/rus таблицы большие, сначала фильтруем)
eng_small = eng[eng['id'].isin(links['eng_id'])]
rus_small = rus[rus['id'].isin(links['rus_id'])]
df = (links
      .merge(rus_small, left_on='rus_id', right_on='id')
      .merge(eng_small, left_on='eng_id', right_on='id'))[['rus', 'eng']].dropna()
df = df.drop_duplicates()
print(len(df))
df.head()


Use a custom tokenizer that can add bos and eos tokens (pass `add_special_tokens=True` when calling the tokenizer to add them).


In [ ]:
class Tokenizer(transformers.GPT2Tokenizer):

    def build_inputs_with_special_tokens(self, token_ids_0, token_ids_1=None):
        if token_ids_1 is None:
            return [self.bos_token_id, *token_ids_0, self.eos_token_id]

        return [self.bos_token_id, *token_ids_0, self.bos_token_id, *token_ids_1, self.eos_token_id]


In [ ]:
tokenizer = Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token_id = tokenizer.eos_token_id

BOS = tokenizer.bos_token_id
EOS = tokenizer.eos_token_id
PAD = 0  # паддинг нулями (mask_zero в embedding)
N_TOKENS = len(tokenizer.get_vocab())


In [ ]:
# смотрим, как токенизируется кириллица (спойлер: плохо, много токенов)
s = 'Я люблю ходить в кино.'
ids = tokenizer.encode(s)
print(s)
print(ids)
print([tokenizer.decode([i]) for i in ids])


In [ ]:
# длины предложений в токенах (на подвыборке, чтобы быстрее)
sample = df.sample(20_000, random_state=42)
len_ru = np.array([len(tokenizer.encode(t)) for t in tqdm.tqdm(sample['rus'])])
len_en = np.array([len(tokenizer.encode(t)) for t in tqdm.tqdm(sample['eng'])])

print('RU quantiles:', np.quantile(len_ru, [0.9, 0.99, 0.999]))
print('EN quantiles:', np.quantile(len_en, [0.9, 0.99, 0.999]))

plt.hist(len_ru, bins=100, range=(0, 150))
plt.hist(len_en, bins=100, range=(0, 150))
plt.legend(['RU', 'EN'])
plt.show()


In [ ]:
# ограничиваем длину: русские фразы длиннее в токенах (BPE под кириллицу не затачивался)
MAX_SRC = 70   # rus, ~99% фраз короче
MAX_TGT = 40   # eng, ~99% фраз короче

# подвыборка, чтобы обучение не заняло вечность
df = df.sample(n=100_000, random_state=42).reset_index(drop=True)


In [ ]:
src_ids, tgt_ids = [], []
for rus, eng in tqdm.tqdm(zip(df['rus'], df['eng']), total=len(df)):
    s = tokenizer(rus, add_special_tokens=True, truncation=True, max_length=MAX_SRC)['input_ids']
    t = tokenizer(eng, add_special_tokens=True, truncation=True, max_length=MAX_TGT)['input_ids']
    if len(s) >= 3 and len(t) >= 3:
        src_ids.append(np.array(s, dtype='int32'))
        tgt_ids.append(np.array(t, dtype='int32'))

len(src_ids)


Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.


In [ ]:
# 500 пар откладываем для теста
N_TEST = 500
test_rus = df['rus'].tolist()[:N_TEST]
test_eng = df['eng'].tolist()[:N_TEST]
train_src = src_ids[N_TEST:]
train_tgt = tgt_ids[N_TEST:]


Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)
* A tensor `(None, P)` -- a masking tensor marking padded tokens with 0


In [ ]:
BATCH = 128

def gen(src_ids, tgt_ids):
    for s, t in zip(src_ids, tgt_ids):
        yield (s, t[:-1]), t[1:], np.ones(len(t) - 1, dtype='float32')

sig = (
    (tf.TensorSpec(shape=(None,), dtype=tf.int32),
     tf.TensorSpec(shape=(None,), dtype=tf.int32)),
    tf.TensorSpec(shape=(None,), dtype=tf.int32),
    tf.TensorSpec(shape=(None,), dtype=tf.float32),
)

train_ds = tf.data.Dataset.from_generator(lambda: gen(train_src, train_tgt), output_signature=sig) \
    .shuffle(20000) \
    .padded_batch(BATCH, padded_shapes=(([None], [None]), [None], [None]),
                  padding_values=((tf.constant(PAD, tf.int32), tf.constant(PAD, tf.int32)),
                                  tf.constant(PAD, tf.int32), tf.constant(0.0))) \
    .prefetch(tf.data.AUTOTUNE)


In [ ]:
(e, d), t, m = next(iter(train_ds))
print('enc:', e.shape, 'dec:', d.shape)
print('target:', t.shape, 'mask:', m.shape)
print()
print('пример:')
print('dec input :', d[0].numpy())
print('target   :', t[0].numpy())
print('mask     :', m[0].numpy())


# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one.


In [ ]:
def get_model(
    units: int,
    n_tokens: int,
    n_labels: int,
    n_stacks: int = 1,
    bidirectional: bool = False,
    name: str | None = None,
    cell_type: type[keras.layers.Layer] = keras.layers.LSTMCell
) -> keras.Model:
    is_lstm = cell_type is keras.layers.LSTMCell

    # энкодер
    enc_in = keras.Input(shape=(None,), dtype='int32', name='enc_in')
    x = keras.layers.Embedding(n_tokens, units, mask_zero=True, name='enc_emb')(enc_in)
    states = []
    for i in range(n_stacks):
        rnn = keras.layers.RNN(cell_type(units), return_sequences=True, return_state=True,
                               name=f'enc_rnn_{i}')
        if bidirectional and i == n_stacks - 1:
            outs = keras.layers.Bidirectional(rnn, name='enc_bidi')(x)
            x = outs[0]
            if is_lstm:
                states = [keras.layers.Concatenate()([outs[1], outs[3]]),
                          keras.layers.Concatenate()([outs[2], outs[4]])]
            else:
                states = [keras.layers.Concatenate()([outs[1], outs[2]])]
        else:
            outs = rnn(x)
            x = outs[0]
            states = list(outs[1:])

    # у двунаправленного энкодера состояние в 2 раза больше, сжимаем до units
    if bidirectional:
        states = [keras.layers.Dense(units, name=f'state_proj_{i}')(s) for i, s in enumerate(states)]

    # декодер
    dec_in = keras.Input(shape=(None,), dtype='int32', name='dec_in')
    d = keras.layers.Embedding(n_tokens, units, mask_zero=True, name='dec_emb')(dec_in)
    for i in range(n_stacks):
        rnn = keras.layers.RNN(cell_type(units), return_sequences=True, name=f'dec_rnn_{i}')
        d = rnn(d, initial_state=states if i == 0 else None)

    d = keras.layers.Dropout(0.2)(d)
    out = keras.layers.Dense(n_labels, name='out')(d)
    return keras.Model([enc_in, dec_in], out, name=name or 'seq2seq')


In [ ]:
UNITS = 256
N_STACKS = 2

model = get_model(UNITS, N_TOKENS, N_TOKENS, n_stacks=N_STACKS,
                  bidirectional=True, cell_type=keras.layers.LSTMCell,
                  name='seq2seq_lstm')
model.summary()


Try to add attention to your model (for example [additive attention](https://keras.io/api/layers/attention_layers/additive_attention/)), does it perform better?

attention не добавлял — не успел разобраться, как его прокидывать в инференс, где декодер вызывается по одному шагу


# Training

Train your model using teacher forcing. The idea is that the model predicts the next token that should follow, so one part of the model (called encoder) reads the text and output some state containing information about the text read. The other part of the model (called decoder) reads an already generated text (or in case of the teacher forcing the expected output) and predicts the next token for each one.


In [ ]:
EPOCHS = 3

model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.Adam(1e-3),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name='acc')],
)

history = model.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ReduceLROnPlateau(monitor='loss', factor=0.5, patience=1),
    ],
)


In [ ]:
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['acc'], label='acc')
plt.xlabel('epoch')
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# Testing

Make a function for text translation. Translate some text and evaluate model performance.

Take note that your model is set for training. During the inference process you will have to use parts of the model independently (including the RNN cells).


In [ ]:
# энкодер: фраза -> финальные состояния (собираем из слоёв обученной модели)
x = model.get_layer('enc_emb')(model.get_layer('enc_in').output)
for i in range(N_STACKS):
    if i == N_STACKS - 1:  # последний слой двунаправленный
        outs = model.get_layer('enc_bidi')(x)
        h = keras.layers.Concatenate()([outs[1], outs[3]])
        c = keras.layers.Concatenate()([outs[2], outs[4]])
        enc_states_out = [model.get_layer('state_proj_0')(h),
                          model.get_layer('state_proj_1')(c)]
    else:
        outs = model.get_layer(f'enc_rnn_{i}')(x)
        x = outs[0]
encoder_model = keras.Model(model.get_layer('enc_in').output, enc_states_out)
encoder_model.summary()


In [ ]:
# декодер: один шаг (токен + состояния -> логиты следующего токена + новые состояния)
tok_in = keras.Input(shape=(1,), dtype='int32')
states_in = []
for i in range(N_STACKS):
    states_in += [keras.Input(shape=(UNITS,)), keras.Input(shape=(UNITS,))]  # h_i, c_i

d = model.get_layer('dec_emb')(tok_in)
new_states = []
for i in range(N_STACKS):
    cell = model.get_layer(f'dec_rnn_{i}').cell
    outs = keras.layers.RNN(cell, return_sequences=True, return_state=True)(
        d, initial_state=[states_in[2 * i], states_in[2 * i + 1]]
    )
    d = outs[0]
    new_states += [outs[1], outs[2]]

logits = model.get_layer('out')(d)[:, -1, :]
decoder_model = keras.Model([tok_in] + states_in, [logits] + new_states)
decoder_model.summary()


In [ ]:
def translate(
    text: str,
    tokenizer: Tokenizer,
    model: keras.Model,
    max_len: int = 20
) -> str:
    '''Predicts `text` translation using the `model`.'''
    ids = tokenizer(text, add_special_tokens=True, truncation=True, max_length=MAX_SRC)['input_ids']
    h, c = encoder_model.predict(np.array([ids]), verbose=0)

    # второй слой декодера стартует с нулей, как при обучении
    states = [h, c, np.zeros((1, UNITS), dtype='float32'), np.zeros((1, UNITS), dtype='float32')]
    tok = np.array([[BOS]], dtype='int32')

    result = []
    for _ in range(max_len):
        logits, *states = decoder_model.predict([tok] + states, verbose=0)
        nxt = int(np.argmax(logits[0]))  # жадно берём самый вероятный токен
        if nxt == EOS:
            break
        result.append(nxt)
        tok = np.array([[nxt]], dtype='int32')
    return tokenizer.decode(result)


In [ ]:
texts = [
    'Я люблю ходить в кино.',
    'Солнце светит на небе.',
    'Это перевод с русского на английский.',
    'Меня зовут Вася.',
    'Я хочу кушать.',
    'Улица. Фонарь. Аптека.',
    'Где находится ближайший магазин?',
]


In [ ]:
for t in texts:
    print('RUS:', t)
    print('ENG:', translate(t, tokenizer, model))
    print()


Оценка качества по BLEU на отложенных парах:


In [ ]:
try:
    import sacrebleu
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sacrebleu'])
    import sacrebleu


In [ ]:
hyps = [translate(t, tokenizer, model) for t in tqdm.tqdm(test_rus[:100])]
bleu = sacrebleu.corpus_bleu(hyps, [test_eng[:100]])
print('BLEU:', round(bleu.score, 2))


In [ ]:
# примеры с теста
for i in range(10):
    print('RUS:', test_rus[i])
    print('REF:', test_eng[i])
    print('HYP:', translate(test_rus[i], tokenizer, model))
    print()


Короткие частотные фразы переводятся нормально, подлиннее — с ошибками. Основная проблема: BPE GPT-2 дробит русский текст на кучу токенов, одно слово — это 5-10 токенов, поэтому энкодеру тяжело. Ещё 3 эпохи на 100k парах — мало, и attention бы помог.
